# 04 — Exposition : normaliser le risque par la densité routière

**Problème corrigé ici.** Les notebooks 02/03 priorisent par le **volume brut** d'accidents. Mais un volume élevé peut venir d'un **lieu dangereux** *ou* simplement d'un **réseau routier dense / très circulé**. Sans dénominateur d'exposition, on ne sait pas trancher — et c'est le principal reproche méthodologique (biais d'exposition).

**Apport.** Pour chaque zone prioritaire, on récupère via **OpenStreetMap** la longueur de routes et le nombre d'intersections dans la cellule H3, puis on calcule des taux :

$$\text{accidents/km} = \frac{\text{accidents}}{\text{km de route}} \qquad \text{charge/km} = \frac{\text{charge pondérée}}{\text{km de route}}$$

On peut alors :
- **re-classer** les zones à exposition égale → les *vrais* points noirs remontent ;
- **distinguer** un *point noir concentré* (forte densité d'accidents/km) d'un *réseau étendu* (volume élevé mais étalé) — ce qui **éclaire les zones « diffus »** de nb3 : beaucoup d'entre elles sont probablement juste très circulées.

**Limite.** OSM donne l'offre routière, pas le trafic réel (AADT). `accidents/km` reste un proxy d'exposition, meilleur que le volume brut mais pas un taux par véhicule·km. Caltrans AADT serait l'étape suivante.

In [1]:
%pip install -q osmnx h3 folium

^C
Note: you may need to restart the kernel to use updated packages.


In [ ]:
import time
import pandas as pd
import numpy as np
import h3
from shapely.geometry import Polygon
import osmnx as ox
import folium

ox.settings.use_cache = True          # cache disque -> ré-exécutions quasi instantanées
ox.settings.log_console = False
print('osmnx', ox.__version__, '| h3', h3.__version__)

In [ ]:
# Zones prioritaires produites par nb3 (03_modele_attribution.ipynb)
reco = pd.read_csv('recos_amenagement_ca.csv')
reco.columns = ['rank', 'h3_cell', 'n_accidents', 'charge', 'avg_severity',
                'dominant_A', 'lift_A', 'equip_present', 'amenagement', 'impact_charge']
print(f'{len(reco)} zones prioritaires chargées')
reco.head()

## Récupération de l'exposition OSM par cellule

Pour chaque cellule H3 : on construit son polygone, on télécharge le réseau routier *drivable* (OSM), et on extrait la longueur totale de voirie et le nombre d'intersections. Le cache OSMnx évite de re-télécharger aux exécutions suivantes.

In [ ]:
def exposure(cell):
    b = h3.cell_to_boundary(cell)                     # [(lat, lng), ...]
    poly = Polygon([(lng, lat) for lat, lng in b])
    try:
        G = ox.graph_from_polygon(poly, network_type='drive', retain_all=True)
        st = ox.stats.basic_stats(G)
        return st['street_length_total'] / 1000.0, int(st['intersection_count'])
    except Exception:
        return np.nan, np.nan

t0 = time.time()
rows = []
for i, cell in enumerate(reco['h3_cell'], 1):
    km, inter = exposure(cell)
    rows.append((km, inter))
    if i % 10 == 0 or i == len(reco):
        print(f'  {i}/{len(reco)} cellules — {time.time()-t0:.0f}s')

reco['street_km'] = [r[0] for r in rows]
reco['n_intersections'] = [r[1] for r in rows]
n_ok = reco['street_km'].notna().sum()
print(f'Exposition récupérée pour {n_ok}/{len(reco)} zones en {time.time()-t0:.0f}s')

## Taux d'exposition et re-classement

On calcule `accidents/km` et `charge/km`, puis on compare le **classement par volume** (nb3) au **classement par charge/km** (exposition). Les zones qui **montent** sont les vrais points noirs masqués par l'exposition ; celles qui **descendent** étaient surtout de gros réseaux.

In [ ]:
d = reco[reco['street_km'].notna()].copy()
d = d[d['street_km'] > 0]
d['acc_per_km']    = d['n_accidents'] / d['street_km']
d['charge_per_km'] = d['charge'] / d['street_km']
d['acc_per_inter'] = d['n_accidents'] / d['n_intersections'].replace(0, np.nan)

# Rangs : volume (nb3) vs exposition (charge/km)
d['rank_volume']   = d['charge'].rank(ascending=False).astype(int)
d['rank_exposure'] = d['charge_per_km'].rank(ascending=False).astype(int)
d['delta_rank']    = d['rank_volume'] - d['rank_exposure']   # >0 = monte une fois normalisé

print('Top 10 par densité d\'accidents (charge/km) :')
top = d.sort_values('charge_per_km', ascending=False).head(10)
show = top[['rank_volume', 'rank_exposure', 'n_accidents', 'street_km',
            'acc_per_km', 'charge_per_km', 'dominant_A']].copy()
show[['street_km', 'acc_per_km', 'charge_per_km']] = show[['street_km', 'acc_per_km', 'charge_per_km']].round(1)
show

In [ ]:
# Classification : point noir concentré vs réseau étendu (seuil = médiane accidents/km)
seuil = d['acc_per_km'].median()
d['type_zone'] = np.where(d['acc_per_km'] >= seuil, 'point noir concentré', 'réseau étendu (volume diffus)')

print(f"Seuil médian : {seuil:.0f} accidents/km")
print(d['type_zone'].value_counts())
print()
# Croisement avec les zones 'diffus' de nb3 : sont-elles juste des réseaux étendus ?
diffus = d[d['dominant_A'] == 'diffus']
if len(diffus):
    print(f"Sur les {len(diffus)} zones 'diffus' de nb3 :")
    print(diffus['type_zone'].value_counts())

In [ ]:
# Plus gros mouvements de classement après normalisation par l'exposition
movers = d.sort_values('delta_rank', ascending=False)
print('Zones qui MONTENT le plus (vrais points noirs masqués par le volume/réseau) :')
up = movers.head(5)[['rank_volume', 'rank_exposure', 'n_accidents', 'street_km', 'acc_per_km', 'dominant_A']]
print(up.round(1).to_string(index=False))
print('\nZones qui DESCENDENT le plus (volume surtout dû à un grand réseau) :')
down = movers.tail(5)[['rank_volume', 'rank_exposure', 'n_accidents', 'street_km', 'acc_per_km', 'dominant_A']]
print(down.round(1).to_string(index=False))

## Livrable enrichi + carte

Export du tableau des zones avec exposition et taux, et carte où la **taille des pastilles = densité d'accidents par km** (l'intensité réelle du risque, pas le simple volume).

In [ ]:
out = d.sort_values('charge_per_km', ascending=False)[[
    'rank_volume', 'rank_exposure', 'delta_rank', 'h3_cell',
    'n_accidents', 'charge', 'street_km', 'n_intersections',
    'acc_per_km', 'charge_per_km', 'type_zone', 'dominant_A', 'amenagement'
]].copy()
for c in ['street_km', 'acc_per_km', 'charge_per_km']:
    out[c] = out[c].round(1)
out.to_csv('zones_exposition_ca.csv', index=False)
print('Export : zones_exposition_ca.csv —', len(out), 'zones')
out.head(15)

In [ ]:
# Carte : pastilles dimensionnées par densité d'accidents/km (risque réel), pas le volume
col_type = {'point noir concentré': '#e74c3c', 'réseau étendu (volume diffus)': '#3498db'}
m = folium.Map(tiles='CartoDB positron')
vmax = d['acc_per_km'].max()
centers = []
for _, r in d.iterrows():
    try:
        c = list(h3.cell_to_latlng(r['h3_cell']))
        centers.append(c)
        radius = 5 + 18 * (r['acc_per_km'] / vmax)
        tip = (f"<b>Vol #{int(r['rank_volume'])} → Expo #{int(r['rank_exposure'])}</b><br>"
               f"{int(r['n_accidents']):,} accidents sur {r['street_km']:.1f} km<br>"
               f"<b>{r['acc_per_km']:.0f} accidents/km</b><br>"
               f"Type : {r['type_zone']}<br>"
               f"Aléa : {r['dominant_A']} — ➜ {r['amenagement']}")
        folium.CircleMarker(location=c, radius=radius, color='#222', weight=1,
                            fill=True, fill_color=col_type[r['type_zone']],
                            fill_opacity=0.85, tooltip=folium.Tooltip(tip)).add_to(m)
    except Exception:
        pass
if centers:
    m.fit_bounds(centers)

items = ''.join(f"<div><span style='background:{v};width:12px;height:12px;display:inline-block;"
                f"margin-right:6px;'></span>{k}</div>" for k, v in col_type.items())
legend = (f"<div style='position:fixed;bottom:30px;left:30px;z-index:9999;background:white;"
          f"padding:10px;border:1px solid #999;font-size:11px;'>"
          f"<b>Type de zone</b><br><i>taille = accidents/km</i>{items}</div>")
m.get_root().html.add_child(folium.Element(legend))
m.save('exposition_map_ca.html')
print('Carte sauvegardée : exposition_map_ca.html')
m

## Lecture

- **accidents/km** mesure l'intensité réelle du risque, à offre routière donnée — c'est un meilleur signal de priorisation que le volume brut.
- Les zones qui **montent** au classement par exposition sont les **vrais points noirs** : peu de voirie, beaucoup d'accidents → défaut localisé.
- Les zones qui **descendent** (souvent des « diffus » de nb3) sont de **grands réseaux circulés** : le volume vient de l'étendue, pas d'un défaut ponctuel → levier plutôt *gestion de trafic / vitesse* qu'aménagement lourd.

**Prochaine étape (②)** : intervalles de confiance sur les deltas de sévérité, et — idéalement — Caltrans AADT pour un vrai taux par trafic.